In [ ]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest, f_classif

from funs import plotDictionary, chi2_independence, highlightTable

# Data Preprocessing

## Data Collection

### Raw Data

In [ ]:
all_trxns = pd.read_csv("../data/all_trxns.csv", dtype={"counterparty": str})

### Exchange Rates
more info in `Currencies.ipynb`


In [ ]:
currency_rates = pd.read_csv(
    "../data/exchange_rates.csv", header=None, names=["ccy", "date", "rate"]
)

## Cleaning and Transformation
Get as many factors as possible
Transform the variables and provide additional features
1. Convert the timestamp to datetime
2. Add the date and the exchange rate
3. Clean and convert the amount to EUR
4. Extract the customer type from the customer id
5. Extract the weekday, month, quarter and hour from the timestamp
6. Replace missing values in the "counterparty_country" column with "unknown"
7. Add "amount_eur" buckets - calculate the thresholds for the equally sized buckets of the amount in EUR

In [ ]:
trxns_data = all_trxns.copy()

trxns_data["timestamp"] = pd.to_datetime(
    trxns_data["timestamp"], infer_datetime_format=True
)

trxns_data["date"] = trxns_data["timestamp"].dt.date
trxns_data = trxns_data.merge(currency_rates, on=["ccy", "date"], how="left")
trxns_data["rate"] = np.where(trxns_data["rate"].isna(), 1, trxns_data["rate"])

trxns_data["amount"] = trxns_data["amount"].apply(
    lambda x: float(re.sub("[^0-9.]", "", x))
)
trxns_data["amount_eur"] = trxns_data["amount"] / trxns_data["rate"]

trxns_data["customer_type"] = trxns_data["customer"].str[0]

trxns_data["weekday"] = trxns_data["date"].apply(lambda x: x.strftime("%A"))
trxns_data["month"] = trxns_data["date"].apply(lambda x: x.strftime("%B"))
trxns_data["quarter"] = trxns_data["date"].apply(
    lambda x: "Q" + str((x.month - 1) // 3 + 1)
)
trxns_data["hour"] = trxns_data["timestamp"].dt.hour
#
trxns_data["counterparty_country"] = np.where(
    trxns_data["counterparty_country"].isna(),
    "unknown",
    trxns_data["counterparty_country"],
)

amount_eur_quantile = np.quantile(trxns_data["amount_eur"], q=np.arange(0, 1.2, 0.2))

trxns_data["amount_eur_bucket"] = pd.cut(
    trxns_data["amount_eur"], bins=amount_eur_quantile, include_lowest=True
)

In [ ]:
print(trxns_data)

# Data Analysis

## Number of Frauds
1. Prepare plot data
2. Plot the data
3. Add the percentage on top of the bars

In [ ]:
plot_data = trxns_data[["fraud_flag"]]
plot_data["count"] = 1
summary = plot_data.groupby("fraud_flag").count()
summary["perc"] = summary["count"] / summary["count"].sum()

fig, ax = plt.subplots()
bars = ax.bar(summary.index, summary["count"], color="#968B89")
ax.set_title("Fraud Flag count")
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.set_ylim(top=summary["count"].max() * 1.2)

for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height * 1.01,
        f'{summary["perc"][i]:.1%}',
        ha="center",
        va="bottom",
    )

plt.show()

In total there is 1.7% probability of a transaction being a fraud.
Next step: identify the variable that is associated with the fraud flag or to find a factor within a variable that is associated with the fraud flag.

## Which variables could be associated with the fraud flag?

In [ ]:
trxns_data.columns

### Chi-squared test

In [ ]:
chi2_independence(trxns_data, "customer_type", "fraud_flag", type="description")

In [ ]:
chi2_independence(trxns_data, "amount_eur_bucket", "fraud_flag", type="description")

In [ ]:
chi2_independence(trxns_data, "weekday", "fraud_flag", type="description")

### Significant Difference Tables
values with the * are the ones that have values significantly different than expected

In [ ]:
highlightTable(trxns_data, "ccy", "fraud_flag")

In [ ]:
highlightTable(trxns_data, "hour", "fraud_flag")

In [ ]:
highlightTable(trxns_data, "amount_eur_bucket", "fraud_flag")

There are a lot of factors within the variables that could be used to identify the fraud flag because the values are significantly different than expected.

### Plots

- black dashed line is the mean value 
- red dashed lines are the values of mean plus 1.5 and 2 standard deviations 
- red solid line is the value of quantile (by default 0.9) plus standard deviation

In [ ]:
plotDictionary(trxns_data, colname="ccy", quantile_threshold=0.9, count_filter=5)

In [ ]:
plotDictionary(trxns_data, colname="customer_type", quantile_threshold=0.9, count_filter=5)

In [ ]:
plotDictionary(trxns_data, colname="counterparty_country", quantile_threshold=0.9, count_filter=5)

In [ ]:
plotDictionary(trxns_data, colname="weekday", quantile_threshold=0.9, count_filter=5)

### ANOVA f-test

With one-hot encoding get all the factors from the variables as separate variables and analyze their association with the fraud flag.

1. Select the features
2. Split the data into features (X) and target (y)
3. Perform one-hot encoding
4. Define the number of top features to select
5. Perform univariate feature selection using ANOVA F-value
6. Get the scores and p-values of each feature
7. Create a DataFrame to store the results
8. Filter out the features with pvalue less or equal to 0.01

In [ ]:
feature_names = [
    "customer_country",
    "counterparty_country",
    "type",
    "ccy",
    "customer_type",
    "weekday",
    "month",
    "quarter",
    "hour",
    "amount_eur_bucket",
]

X = trxns_data[feature_names]
y = trxns_data["fraud_flag"]

X_encoded = pd.get_dummies(X, columns=feature_names)

# Chronological split first: SelectKBest must see only the train distribution.
from funs import chronological_split

X_train, X_test, y_train, y_test = chronological_split(
    X_encoded, y, timestamp=trxns_data["timestamp"], test_size=0.2, random_state=42
)

k = 50

selector = SelectKBest(f_classif, k=k)
selector.fit(X_train, y_train)

scores = selector.scores_
pvalues = selector.pvalues_

results_df = pd.DataFrame(
    {"feature": X_train.columns, "score": scores, "pvalue": pvalues}
)
results_df = results_df.sort_values(by="score", ascending=False)

results_df = results_df[results_df["pvalue"] <= 0.01]
print(results_df.head(k))